In [ ]:
import chromadb

In [6]:
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
import chromadb.utils.embedding_functions as embedding_functions
openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key_env_var="OPENAI_API_KEY",
    model_name="text-embedding-3-small"
)

In [8]:
client = chromadb.PersistentClient(path="./chroma_db")

collection = client.get_or_create_collection(name="rag", embedding_function=openai_ef)

In [9]:
from langchain_community.document_loaders.pdf import PyPDFLoader

loader = PyPDFLoader("data/text.pdf")

document = loader.load()


In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " "]

)
doc_splits = text_splitter.split_documents(document)

In [11]:
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings

vector_store = Chroma.from_documents(
    documents=doc_splits, 
    collection_name="rag", 
    client=client,
    embedding=OpenAIEmbeddings(model="text-embedding-3-small")
)

In [13]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

In [14]:
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI


# Data model
class GradeChunks(BaseModel):
    """Binary score for relevance check on retrieved chunks."""

    binary_score: str = Field(
        description="Chunks are relevant to the question, 'yes' or 'no'"
    )


# LLM with function call
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
structured_llm_grader = llm.with_structured_output(GradeChunks)

# Prompt
system = """You are a grader assessing relevance of a retrieved chunk to a user question. \n 
    It does not need to be a stringent test. The goal is to filter out erroneous retrievals. \n
    If the chunk contains keyword(s) or semantic meaning related to the user question, grade it as relevant. \n
    Give a binary score 'yes' or 'no' score to indicate whether the chunk is relevant to the question."""
grade_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "Retrieved chunk: \n\n {document} \n\n User question: {question}"),
    ]
)

retrieval_grader = grade_prompt | structured_llm_grader
question = "Який порт використовується для підключення до API у версії 2026.4?"
docs = retriever.invoke(question)

if not docs:
    print("No chunks retrieved.")
else:
    doc_txt = docs[0].page_content
    print(retrieval_grader.invoke({"question": question, "document": doc_txt}))

binary_score='yes'


In [15]:
### Generate

from langchain_core.output_parsers import StrOutputParser

# Prompt
prompt = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks.
Use the following retrieved context to answer the question.
If you don't know the answer, just say you don't know.

Question: {question}
Context:
{context}

Answer:"""
)

# LLM
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)


# Post-processing
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


# Chain
rag_chain = prompt | llm | StrOutputParser()

# Run
generation = rag_chain.invoke({"context": format_docs(docs), "question": question})
print(generation)

Основним портом для підключення до API у версії 2026.4 є порт 8080.


In [16]:
### Hallucination Grader


# Data model
class GradeHallucinations(BaseModel):
    """Binary score for hallucination present in generation answer."""

    binary_score: str = Field(
        description="Answer is grounded in the facts, 'yes' or 'no'"
    )


# LLM with function call
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
structured_llm_grader = llm.with_structured_output(GradeHallucinations)

# Prompt
system = """You are a grader assessing whether an LLM generation is grounded in / supported by a set of retrieved facts. \n 
     Give a binary score 'yes' or 'no'. 'Yes' means that the answer is grounded in / supported by the set of facts."""
hallucination_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "Set of facts: \n\n {documents} \n\n LLM generation: {generation}"),
    ]
)

hallucination_grader = hallucination_prompt | structured_llm_grader
hallucination_grader.invoke({"documents": docs, "generation": generation})

GradeHallucinations(binary_score='yes')

In [17]:
### Answer Grader


# Data model
class GradeAnswer(BaseModel):
    """Binary score to assess answer addresses question."""

    binary_score: str = Field(
        description="Answer addresses the question, 'yes' or 'no'"
    )


# LLM with function call
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
structured_llm_grader = llm.with_structured_output(GradeAnswer)

# Prompt
system = """You are a grader assessing whether an answer addresses / resolves a question \n 
     Give a binary score 'yes' or 'no'. Yes' means that the answer resolves the question."""
answer_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "User question: \n\n {question} \n\n LLM generation: {generation}"),
    ]
)

answer_grader = answer_prompt | structured_llm_grader
answer_grader.invoke({"question": question, "generation": generation})

GradeAnswer(binary_score='yes')

In [55]:
### Question Re-writer

# LLM
llm = ChatOpenAI(model="gpt-3.5-turbo-0125", temperature=0)

# Prompt
system = """You a question re-writer that converts an input question to a better version that is optimized \n 
     for vectorstore retrieval. Look at the input and try to reason about the underlying semantic intent / meaning."""
re_write_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        (
            "human",
            "Here is the initial question: \n\n {question} \n Formulate an improved question.",
        ),
    ]
)

question_rewriter = re_write_prompt | llm | StrOutputParser()
question_rewriter.invoke({"question": question})

'Which port is used for connecting to the API in version 2026.4?'

In [61]:
llm = ChatOpenAI(model="gpt-3.5-turbo-0125", temperature=0)

# Prompt
system = """You are a response writer that answers a question using retrieved context. If the context does not contain enough information, answer that you do not know."""
response_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        (
            "human",
            "Context: \n\n {context} \n\n Question: \n\n {question} \n\n Formulate a response.",
        ),
    ]
)

response_generator = response_prompt | llm | StrOutputParser()
response_generator.invoke({"question": question, "context": ""})

'I do not have enough information to provide an answer to this question.'

In [57]:
from typing import Any, List

from typing_extensions import TypedDict


class GraphState(TypedDict):
    """
    Represents the state of our graph.

    Attributes:
        question: question
        generation: LLM generation
        documents: list of documents
        counter: generation attempts counter
        transform_counter: query transform attempts counter
    """

    question: str
    generation: str
    documents: List[Any]
    counter: int
    transform_counter: int

In [62]:
### Nodes

MAX_RETRIES = 2
MAX_TRANSFORMS = 2

def retrieve(state):
    """
    Retrieve documents

    Args:
        state (dict): The current graph state

    Returns:
        state (dict): New key added to state, documents, that contains retrieved documents
    """
    print("---RETRIEVE---")
    question = state["question"]
    counter = state.get("counter", 0)
    transform_counter = state.get("transform_counter", 0)

    # Retrieval
    documents = retriever.invoke(question)
    return {
        "documents": documents,
        "question": question,
        "counter": counter,
        "transform_counter": transform_counter,
    }


def generate(state):
    """
    Generate answer

    Args:
        state (dict): The current graph state

    Returns:
        state (dict): New key added to state, generation, that contains LLM generation
    """
    print("---GENERATE---")
    question = state["question"]
    documents = state["documents"]
    counter = state.get("counter", 0) + 1
    transform_counter = state.get("transform_counter", 0)

    # RAG generation
    generation = rag_chain.invoke({"context": format_docs(documents), "question": question})
    return {
        "documents": documents,
        "question": question,
        "generation": generation,
        "counter": counter,
        "transform_counter": transform_counter,
    }


def grade_documents(state):
    """
    Determines whether the retrieved documents are relevant to the question.

    Args:
        state (dict): The current graph state

    Returns:
        state (dict): Updates documents key with only filtered relevant documents
    """

    print("---CHECK DOCUMENT RELEVANCE TO QUESTION---")
    question = state["question"]
    documents = state["documents"]
    counter = state.get("counter", 0)
    transform_counter = state.get("transform_counter", 0)

    # Score each doc
    filtered_docs = []
    for d in documents:
        score = retrieval_grader.invoke(
            {"question": question, "document": d.page_content}
        )
        grade = score.binary_score
        if grade == "yes":
            print("---GRADE: DOCUMENT RELEVANT---")
            filtered_docs.append(d)
        else:
            print("---GRADE: DOCUMENT NOT RELEVANT---")
            continue
    return {
        "documents": filtered_docs,
        "question": question,
        "counter": counter,
        "transform_counter": transform_counter,
    }


def transform_query(state):
    """
    Transform the query to produce a better question.

    Args:
        state (dict): The current graph state

    Returns:
        state (dict): Updates question key with a re-phrased question
    """

    print("---TRANSFORM QUERY---")
    question = state["question"]
    documents = state["documents"]
    counter = state.get("counter", 0)
    transform_counter = state.get("transform_counter", 0) + 1

    # Re-write question
    better_question = question_rewriter.invoke({"question": question})
    return {
        "documents": documents,
        "question": better_question,
        "counter": counter,
        "transform_counter": transform_counter,
    }


### Edges


def decide_to_generate(state):
    """
    Determines whether to generate an answer, transform a query, or stop with fallback answer.

    Args:
        state (dict): The current graph state

    Returns:
        str: Decision for next node to call
    """

    print("---ASSESS GRADED DOCUMENTS---")
    filtered_documents = state["documents"]
    transform_counter = state.get("transform_counter", 0)

    if not filtered_documents:
        if transform_counter >= MAX_TRANSFORMS:
            print("---DECISION: MAX TRANSFORM ATTEMPTS REACHED, RETURN UNKNOWN---")
            return "unknown"
        print(
            "---DECISION: ALL DOCUMENTS ARE NOT RELEVANT TO QUESTION, TRANSFORM QUERY---"
        )
        return "transform_query"
    else:
        # We have relevant documents, so generate answer
        print("---DECISION: GENERATE---")
        return "generate"


def grade_generation_v_documents_and_question(state):
    """
    Determines whether the generation is grounded in the document and answers question.

    Args:
        state (dict): The current graph state

    Returns:
        str: Decision for next node to call
    """
    print("---CHECK HALLUCINATIONS---")
    question = state["question"]
    documents = state["documents"]
    generation = state["generation"]
    counter = state.get("counter", 0)

    score = hallucination_grader.invoke(
        {"documents": documents, "generation": generation}
    )
    grade = score.binary_score

    # Check hallucination
    if grade == "yes":
        print("---DECISION: GENERATION IS GROUNDED IN DOCUMENTS---")
        # Check question-answering
        print("---GRADE GENERATION vs QUESTION---")
        score = answer_grader.invoke({"question": question, "generation": generation})
        grade = score.binary_score
        if grade == "yes":
            print("---DECISION: GENERATION ADDRESSES QUESTION---")
            return "useful"
        else:
            print("---DECISION: GENERATION DOES NOT ADDRESS QUESTION---")
            return "not useful"
    else:
        if counter >= MAX_RETRIES:
            print("---DECISION: MAX RETRIES REACHED---")
            return "max retries"
        print("---DECISION: GENERATION IS NOT GROUNDED IN DOCUMENTS, RE-TRY---")
        return "not supported"


def generate_unknown(state):
    """
    Returns a fallback answer when retry/transform limits are reached.

    Args:
        state (dict): The current graph state

    Returns:
        dict: Updated graph state with fallback generation
    """
    print("---RETURN UNKNOWN---")
    question = state["question"]
    documents = state.get("documents", [])
    counter = state.get("counter", 0)
    transform_counter = state.get("transform_counter", 0)
    response = response_generator.invoke({"question": question, "context": ""})
    return {
        "question": question,
        "documents": documents,
        "generation": response,
        "counter": counter,
        "transform_counter": transform_counter,
    }

In [63]:
from langgraph.graph import END, StateGraph, START

workflow = StateGraph(GraphState)

# Define the nodes
workflow.add_node("retrieve", retrieve)  # retrieve
workflow.add_node("grade_documents", grade_documents)  # grade documents
workflow.add_node("generate", generate)  # generate
workflow.add_node("transform_query", transform_query)  # transform_query
workflow.add_node("unknown", generate_unknown)  # return unknown answer

# Build graph
workflow.add_edge(START, "retrieve")
workflow.add_edge("retrieve", "grade_documents")
workflow.add_conditional_edges(
    "grade_documents",
    decide_to_generate,
    {
        "transform_query": "transform_query",
        "generate": "generate",
        "unknown": "unknown",
    },
)
workflow.add_edge("transform_query", "retrieve")
workflow.add_conditional_edges(
    "generate",
    grade_generation_v_documents_and_question,
    {
        "not supported": "generate",
        "useful": END,
        "not useful": "transform_query",
        "max retries": "unknown",
    },
)
workflow.add_edge("unknown", END)

# Compile
app = workflow.compile()

In [65]:
from pprint import pprint

# Run
inputs = {
    "question": "Що мені робити, якщо база даних відповідає занадто повільно?",
    "counter": 0,
    "transform_counter": 0,
}
for output in app.stream(inputs):
    for key, value in output.items():
        # Node
        pprint(f"Node '{key}':")
    pprint("\n---\n")

# Final generation
pprint(value.get("generation"))
pprint(f"Total generations: {value.get('counter', 0)}")
pprint(f"Total transforms: {value.get('transform_counter', 0)}")

---RETRIEVE---
"Node 'retrieve':"
'\n---\n'
---CHECK DOCUMENT RELEVANCE TO QUESTION---
---GRADE: DOCUMENT RELEVANT---
---GRADE: DOCUMENT RELEVANT---
---GRADE: DOCUMENT RELEVANT---
---ASSESS GRADED DOCUMENTS---
---DECISION: GENERATE---
"Node 'grade_documents':"
'\n---\n'
---GENERATE---
---CHECK HALLUCINATIONS---
---DECISION: GENERATION IS GROUNDED IN DOCUMENTS---
---GRADE GENERATION vs QUESTION---
---DECISION: GENERATION ADDRESSES QUESTION---
"Node 'generate':"
'\n---\n'
('Перевірте статус сервісу ChromaDB та очистіть кеш семантичного роутера. '
 'Перевірте ліміти токенів у вашому API-шлюзі.')
'Total generations: 1'
'Total transforms: 0'
